#P1: DESMUNTANT LA MENTIDA VISUAL

PREPOCESSAMENT DE DADES

In [72]:
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", None)

In [73]:
df = pd.read_csv("dades/dades_grafic.csv")
df.head()

,Indicator Name,Indicator Code,Country Name,Country Code,Year,Value,Disaggregation
0,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2025,63.674,"female, Modeled, 15+"
1,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2024,63.643,"female, Modeled, 15+"
2,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2023,63.593,"female, Modeled, 15+"
3,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2022,59.831,"female, Modeled, 15+"
4,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2021,59.454,"female, Modeled, 15+"


In [74]:
df.columns

Index(['Indicator Name', 'Indicator Code', 'Country Name', 'Country Code',
       'Year', 'Value', 'Disaggregation'],
      dtype='object')

El format del csv original era peculiar (ja que hi ha diferents tipus de Indicator Name), per tant, primer faig una adaptació. Se selecciona només les dades del 2023 i es guarda per cada país el % de dones i el % d'homes. 
Per tant, el primer que faig és començar a processar el dataframe però treballant només amb les dades del 2023, ja que és l'any en que es basa el gràfic.

In [75]:
df_2023 = df[df["Year"] == 2023]
df_2023.shape

(2841, 7)

El primer que volia comprovar era si el nombre de files era equiparable al nombre de països. Mirant per sobre el csv em vaig adonar que hi havia diferents entrades segons si es tractava del "Labor force participation rate" male i female. Per tant, el nombre de països hauria de ser la meitat del nombre de files. 


In [76]:
rows = len(df_2023)
total_indicator_names = len(df_2023["Indicator Name"].unique())
print(f"Hi ha {total_indicator_names} Indicator Names diferents")
total_countries = len(df_2023["Country Name"].unique())
print(f"Hi ha {total_countries} països i {rows} files")


Hi ha 15 Indicator Names diferents
Hi ha 236 països i 2841 files


Tal i com s'observa, hi ha una gran diferència respecte el nombre d'entrades i el nombre de països del dataset. 
A partir d'aquí, vaig decidir agrupar pels diferents "Indicator Name", per veure si en cadascun hi havia aquests 262 països

També, comprovo que hi hagi el mateix nombre de Country Names que de Country Codes. D'aquesta manera, comprovo que no hi hagi més d'un codi associat a cada país de daltaframe .

In [77]:
if(len(df_2023["Country Name"].unique())==len(df_2023["Country Code"].unique())):
    print("Cada país té un codi associat.")
else:
    print("Hi ha discrepàncies entre el nombre de països i el nombre de codis")

Cada país té un codi associat.


Comprovo que realment un mateix country code només està associat a un país

In [93]:

paisos_codis = df.groupby("Country Name")["Country Code"].nunique()
if(max(paisos_codis)>1):
    print("Hi ha països associats a més d'un codi")
else:
    print("Cada país té un únic codi associat.")

Cada país té un únic codi associat.


In [79]:
indicador_names = df_2023.groupby("Indicator Name")
indicador_names.size()

Indicator Name
Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)                     231
Labor force participation rate for ages 15-24, female (%) (national estimate)                        127
Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)                       231
Labor force participation rate for ages 15-24, male (%) (national estimate)                          127
Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)                      231
Labor force participation rate for ages 15-24, total (%) (national estimate)                         127
Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)      231
Labor force participation rate, female (% of female population ages 15+) (national estimate)         127
Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)    231
Labor force participation rate, male (% 

En el gràfic posa escrit "Figures are rounded and based on the International Labour Organization's estimates as of 2023", per tant, treballaré amb aquells que indiquin "modeled ILO estimate". A partir d'aquí, cal averiguar quin és el rang d'edat (15-24, 15-64 o 15+) que es mostra al gràfic (ja que no està indicat).
Per comprovar quin rang d'edat s'utilitza en el gràfic, comparo les dades d'Afganistan per als diferents intervals disponibles. El gràfic mostra un 100% de participació laboral masculina i un 5% femenina, per tant, busco quin dels indicadors s'aproxima a aquests valors.

In [81]:
afghanistan=df_2023[df_2023["Country Name"] == "Afghanistan"]
afghanistan=afghanistan[afghanistan["Indicator Name"].str.contains("ILO estimate")]
afghanistan = afghanistan[["Indicator Name","Value"]]
afghanistan

,Indicator Name,Value
1694,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",5.145
15611,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",70.074
29527,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",37.588
43670,"Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)",6.013
56953,"Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)",68.598
70236,"Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)",38.006
83544,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567


Tal i com s'observa, en el % d'homes en cap de les tres opcions coincideix amb el 100% que es mostra el gràfic, ja que els valors se situen entre el 68,598% i el 71,384%.
En canvi, en el % de dones, sí que hi ha el 5% en les entrades on l'edat és de 15-64 o de 15+. 
Per tant, miro pels 12 paísos del gràfic tots els valors, per veure si és un cas aïllat o és per tots els països.

In [ ]:
#Tots els noms dels països del gràfic i del CSV no coincideixen, però amb agafant els nous noms ja s'agafen
paisos = [
    "Afghanistan",
    "Iraq",
    "Pakistan",
    "Oman",
    "Iran, Islamic Rep.",
    "Yemen, Rep.",
    "Egypt, Arab Rep.",
    "Algeria",
    "Syrian Arab Republic",
    "Jordan",
    "Morocco",
    "Saudi Arabia"
]
df_12 = df_2023[df_2023["Country Name"].isin(paisos)]
df_12 = df_12[df_12["Indicator Name"].str.contains("modeled ILO estimate")]
df_12_15 = df_12[df_12["Indicator Name"].str.contains("population ages 15-64",regex=False)]
df_12_15 = df_12_15.sort_values(["Country Name", "Indicator Name"])
df_12_15[["Country Name", "Indicator Name", "Value"]]


,Country Name,Indicator Name,Value
83544,Afghanistan,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,Afghanistan,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,Afghanistan,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567
83616,Algeria,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",15.834
92026,Algeria,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",73.378
100436,Algeria,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",45.346
85308,"Egypt, Arab Rep.","Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",19.830
93718,"Egypt, Arab Rep.","Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",74.050
102128,"Egypt, Arab Rep.","Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",47.332
86352,"Iran, Islamic Rep.","Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",15.650


In [98]:
afghanistan=df_2023[df_2023["Country Name"] == "Afghanistan"]
afghanistan=afghanistan[afghanistan["Indicator Name"].str.contains("ILO estimate")]
afghanistan = afghanistan[["Indicator Name","Value"]]
afghanistan

,Indicator Name,Value
1694,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",5.145
15611,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",70.074
29527,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",37.588
43670,"Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)",6.013
56953,"Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)",68.598
70236,"Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)",38.006
83544,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567


Tal i com s'observa, alguns valors coincideixen, però no tots. He provat agafant els diferents tipus de rangs d'edats i al final he escollit el de 15+.
Són amb aquests valors que contrueixo un nou dataframe transposat per poder després fer el gràfic. 

In [ ]:
df_grafic = df_12_15.pivot(
    index="Country Name",
    columns="Indicator Name",
    values="Value"
)
df_grafic = df_grafic.rename(columns={
    "Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)": "Female",
    "Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)": "Male",
    "Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)": "Total"
})

df_grafic = df_grafic.reset_index()
df_grafic.columns.name = None
df_grafic

,Country Name,Female,Male,Total
0,Afghanistan,5.359,71.384,38.567
1,Algeria,15.834,73.378,45.346
2,"Egypt, Arab Rep.",19.830,74.050,47.332
3,"Iran, Islamic Rep.",15.650,73.892,45.302
4,Iraq,11.505,75.346,43.445
5,Jordan,17.057,66.084,42.620
6,Morocco,21.374,73.674,47.784
7,Oman,31.945,89.774,70.389
8,Pakistan,25.475,82.873,54.686
9,Saudi Arabia,35.617,84.185,66.724


Alguns dels valors són similars, però sí que per exemple en països com Afganistan, es passa d'un 100% a un 70%.
Com que això m'ha sorprès, he decidit fer una nova taula comparativa. En aquesta hi ha els valors del gràfic escrits manualment, per detectar i demostrar aquestes discrepàncies. 

In [112]:
grafic_original = pd.DataFrame({
    "Country Name": paisos,
    "Grafic_Female": [5, 11, 25, 32, 14, 5, 17, 17, 14, 14, 20, 35],
    "Grafic_Male": [100, 92, 81, 88, 71, 60, 71, 67, 64, 63, 68, 80]
})
comparacio = df_grafic.merge(grafic_original, on="Country Name")
#Comparo els valors del gràfic amb els del dataset per veure'n les diferències.
comparacio["Dif_Female"] = abs(comparacio["Grafic_Female"] - comparacio["Female"])
comparacio["Dif_Male"] = abs(comparacio["Grafic_Male"] - comparacio["Male"])
comparacio = comparacio[["Country Name", "Female", "Grafic_Female", "Dif_Female", "Male", "Grafic_Male","Dif_Male"]]
comparacio


,Country Name,Female,Grafic_Female,Dif_Female,Male,Grafic_Male,Dif_Male
0,Afghanistan,5.359,5,0.359,71.384,100,28.616
1,Algeria,15.834,17,1.166,73.378,67,6.378
2,"Egypt, Arab Rep.",19.830,17,2.830,74.050,71,3.050
3,"Iran, Islamic Rep.",15.650,14,1.650,73.892,71,2.892
4,Iraq,11.505,11,0.505,75.346,92,16.654
5,Jordan,17.057,14,3.057,66.084,63,3.084
6,Morocco,21.374,20,1.374,73.674,68,5.674
7,Oman,31.945,32,0.055,89.774,88,1.774
8,Pakistan,25.475,25,0.475,82.873,81,1.873
9,Saudi Arabia,35.617,35,0.617,84.185,80,4.185


Tal i com s'observa, alguns valors coincideixen aproximadament, però alguns altres presenten diferències importants. Per exemple, a Iraq, el gràfic mostra un 92% d'homes, mentre que en el CSV es donen valors de 75,346% per als homes i 11,505% per a les dones. Passa igual, com ja he mencionat, amb Afganistan, es passa d'un 100% del gràfic a un 71,384%.
Per tant, a partir d'ara treballaré amb el dataframe corregit, que correspon a: df_grafic

Considerant que els valors del gràfic no està clar d'on provenen, he volgut comprovar que els 12 països seleccionats siguin realment els 12 països amb  major diferència entre la participació laboral masculina i femenina segons les dades de l'ILO.


In [113]:
df_grafic["Difference"] = df_grafic["Male"] - df_grafic["Female"]
df_top12 = df_grafic.sort_values("Difference",ascending=False).head(12)

df_top12[["Country Name", "Female", "Male", "Difference"]]

,Country Name,Female,Male,Difference
0,Afghanistan,5.359,71.384,66.025
4,Iraq,11.505,75.346,63.841
3,"Iran, Islamic Rep.",15.650,73.892,58.242
7,Oman,31.945,89.774,57.829
1,Algeria,15.834,73.378,57.544
11,"Yemen, Rep.",4.938,62.400,57.462
8,Pakistan,25.475,82.873,57.398
2,"Egypt, Arab Rep.",19.830,74.050,54.220
6,Morocco,21.374,73.674,52.300
10,Syrian Arab Republic,13.790,64.536,50.746


Tal i com s'observa, són els mateixos 12 països mostrats en el gràfic. 
El següent pas és mirar quina és la millor manera d'ordenar les dades de la gràfica. 
En el gràfic original estan ordenats tal que aixi (Afghanistan, Iraq, Pakistan, Oman, Iran, Yemen, Egypt, Algeria, Syria, Jordan, Morocco i Saudi Arabia), en canvi, pandas ordena els valors de forma alfabètica. 
En el gràfic tampoc estan ordenats de cap manera específica els països, ni per major/menor percentatge d'homes, ni pel major/menor percentatge de dones. 
Per tant, es mantindrà l'ordre original del gràfic per facilitar-ne la comparació. TODO: PREGUNTAR!!!

